# 03 Change detection on Hi-UCD mini

1. Unzip Hi-UCD mini and work out its folder layout
2. Fine-tune the U-Net on Hi-UCD's 9 classes, starting from the UAVid weights
3. Segment both years and compare them pixel by pixel (post-classification change detection)
4. Score against Hi-UCD's change labels

Put the Hi-UCD mini zip file(s) in `MyDrive/drone-mapping/data/hiucd/` before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/drone-mapping'
HIUCD_DRIVE = f'{DRIVE}/data/hiucd'          # Hi-UCD mini zip file(s) go here
OUT = f'{DRIVE}/outputs/change'
for d in (HIUCD_DRIVE, f'{DRIVE}/models', OUT):
    os.makedirs(d, exist_ok=True)

!pip install -q segmentation-models-pytorch
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Unzip and inspect

In [ ]:
import glob, shutil

ROOT = '/content/hiucd'                        # local disk is much faster than Drive for training
archives = sorted(glob.glob(f'{HIUCD_DRIVE}/*.zip') + glob.glob(f'{HIUCD_DRIVE}/*.tar*'))
print('Archives found:', [os.path.basename(a) for a in archives])

if not os.path.exists(ROOT):
    os.makedirs(ROOT)
    if archives:
        for a in archives:
            if a.endswith('.zip'):
                !unzip -q -o "{a}" -d {ROOT}
            else:
                !tar -xf "{a}" -C {ROOT}
    else:
        # Maybe the dataset was saved to Drive as an extracted folder
        shutil.copytree(HIUCD_DRIVE, ROOT, dirs_exist_ok=True)
print('Data in', ROOT)

In [ ]:
import re
import cv2
import numpy as np

IMG_EXT = ('.png', '.tif', '.tiff', '.jpg', '.jpeg')

dirs = {}
for d, _, files in os.walk(ROOT):
    f = sorted(x for x in files if x.lower().endswith(IMG_EXT))
    if f:
        dirs[d] = f
assert dirs, f'No images found under {ROOT}. Check the archive contents.'

print('Folders with images:')
for d, f in sorted(dirs.items()):
    print(f'  {os.path.relpath(d, ROOT):55s} {len(f):6d} files   e.g. {f[0]}')


def guess_role(d):
    """Guess (split, kind, year) of an image folder from its path."""
    p = os.path.relpath(d, ROOT).lower().replace(os.sep, '/')
    parts = [x for x in re.split(r'[/_\-\s.]', p) if x]
    split = 'train'
    for name, keys in (('val', ('val', 'valid', 'validation')), ('test', ('test',)), ('train', ('train', 'training'))):
        if any(x in keys for x in parts):
            split = name
            break
    years = re.findall(r'20\d\d', p)
    if 'change' in p or 'cd' in parts:
        kind = 'change'
    elif any(k in parts for k in ('mask', 'masks', 'label', 'labels', 'gt', 'sem', 'semantic', 'annotation', 'annotations')):
        kind = 'label'
    else:
        kind = 'image'
    year = None if kind == 'change' else (years[-1] if years else None)
    return split, kind, year


roles = {}
for d in dirs:
    roles.setdefault(guess_role(d), []).append(d)

print('\nGuessed roles (split, kind, year):')
for key in sorted(roles, key=str):
    print(f'  {str(key):32s} -> {[os.path.relpath(x, ROOT) for x in roles[key]]}')
dupes = {k: v for k, v in roles.items() if len(v) > 1}
if dupes:
    print('\nWARNING: more than one folder for the same role. Set ROLE_OVERRIDE below:', list(dupes))

Check the guessed roles above. If any are wrong, fill in `ROLE_OVERRIDE` with
`(split, kind, year): 'relative/folder/path'` and rerun the cell. `kind` is `image`, `label` or `change`;
`year` is `None` for change folders.

In [ ]:
ROLE_OVERRIDE = {
    # ('train', 'image', '2018'): 'train/image/2018',
}

R = {k: v[0] for k, v in roles.items()}
for k, v in ROLE_OVERRIDE.items():
    R[k] = os.path.join(ROOT, v)

YEARS = sorted({y for (s, k, y) in R if k == 'image' and y})
assert len(YEARS) >= 2, f'Need images from two years, found {YEARS}. Use ROLE_OVERRIDE.'
Y1, Y2 = YEARS[0], YEARS[-1]
print(f'Comparing {Y1} -> {Y2}')


def by_stem(folder):
    return {os.path.splitext(f)[0]: os.path.join(folder, f) for f in dirs.get(folder, [])} if folder else {}


def build_split(split):
    i1, i2 = by_stem(R.get((split, 'image', Y1))), by_stem(R.get((split, 'image', Y2)))
    l1, l2 = by_stem(R.get((split, 'label', Y1))), by_stem(R.get((split, 'label', Y2)))
    ch = by_stem(R.get((split, 'change', None)))
    packed = R.get((split, 'label', None))
    if packed and not (l1 or l2):
        # Full Hi-UCD packs labels into one PNG: channel 0 = year 1, 1 = year 2, 2 = change
        pk = by_stem(packed)
        l1, l2 = {s: p + '#0' for s, p in pk.items()}, {s: p + '#1' for s, p in pk.items()}
        ch = ch or {s: p + '#2' for s, p in pk.items()}
    pairs = []
    for s in sorted(set(i1) & set(i2)):
        pairs.append({'stem': s, 'img1': i1[s], 'img2': i2[s],
                      'lab1': l1.get(s), 'lab2': l2.get(s), 'chg': ch.get(s)})
    return pairs


SPLITS = {s: build_split(s) for s in ('train', 'val', 'test')}
for s, p in SPLITS.items():
    print(f'{s:5s}: {len(p):5d} pairs | with labels {sum(1 for x in p if x["lab1"] and x["lab2"]):5d} '
          f'| with change labels {sum(1 for x in p if x["chg"]):5d}')

In [ ]:
# Hi-UCD label schemes (from the dataset's GitHub page)
CLASSES = ['Water', 'Grass', 'Building', 'Greenhouse', 'Road', 'Bridge', 'Others', 'BareLand', 'Woodland']
NUM_CLASSES = len(CLASSES)                                   # file ids 1..9 become 0..8, 0 = unlabeled
SEM_RGB = [(255, 255, 255), (0, 153, 255), (202, 255, 122), (230, 0, 0), (230, 0, 255),
           (255, 230, 0), (255, 181, 197), (0, 255, 230), (175, 122, 255), (26, 255, 0)]
CHG_RGB = [(255, 255, 255), (0, 0, 0), (220, 0, 0)]          # unlabeled, unchanged, changed
PALETTE = np.array(SEM_RGB[1:], np.uint8)                   # colours for our 0..8 ids
IGNORE = 255
PIXEL_M = 0.1                                               # Hi-UCD ground resolution, metres


def rgb_to_index(rgb, colours):
    out = np.full(rgb.shape[:2], IGNORE, np.uint8)
    for i, c in enumerate(colours):
        out[np.all(rgb == c, axis=-1)] = i
    return out


def read_raw(path):
    """Reads a label file. 'file.png#1' means channel 1 of a packed label PNG."""
    channel = None
    if '#' in os.path.basename(path):
        path, channel = path.rsplit('#', 1)
        channel = int(channel)
    m = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if m is None:
        raise FileNotFoundError(path)
    if m.ndim == 3:
        m = cv2.cvtColor(m[..., :3], cv2.COLOR_BGR2RGB)
    return m[..., channel] if channel is not None else m


def read_image(path):
    return cv2.cvtColor(cv2.imread(path, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)


def read_sem(path):
    m = read_raw(path)
    if m.ndim == 3:
        m = rgb_to_index(m, SEM_RGB)
    m = m.astype(np.int16)
    out = np.full(m.shape, IGNORE, np.uint8)
    ok = (m >= 1) & (m <= NUM_CLASSES)
    out[ok] = m[ok] - 1
    return out


CHG_SCHEME = None   # set by detect_change_scheme()


def detect_change_scheme(paths, n=50):
    """Hi-UCD uses 0 unlabeled / 1 unchanged / 2 changed, but some copies store 0/255 masks."""
    vals = set()
    for p in paths[:n]:
        m = read_raw(p)
        if m.ndim == 3:
            return 'rgb'
        vals |= set(np.unique(m).tolist())
    if 2 in vals:
        return 'index'
    if vals <= {0, 255} and 255 in vals:
        return 'binary255'
    return 'index'


def read_chg(path):
    """Returns 0 = unchanged, 1 = changed, 255 = unlabeled."""
    m = read_raw(path)
    if CHG_SCHEME == 'rgb':
        idx = rgb_to_index(m, CHG_RGB)
        return np.select([idx == 1, idx == 2], [0, 1], IGNORE).astype(np.uint8)
    if CHG_SCHEME == 'binary255':
        return (m > 127).astype(np.uint8)
    return np.select([m == 1, m == 2], [0, 1], IGNORE).astype(np.uint8)


# Inspect one labelled pair
all_pairs = [p for s in SPLITS.values() for p in s]
chg_paths = [p['chg'] for p in all_pairs if p['chg']]
CHG_SCHEME = detect_change_scheme(chg_paths) if chg_paths else None
print('Change label scheme:', CHG_SCHEME)

ex = next((p for p in all_pairs if p['lab1'] and p['chg']), all_pairs[0])
img = read_image(ex['img1'])
print('Image', img.shape, img.dtype)
if ex['lab1']:
    raw = read_raw(ex['lab1'])
    vals = np.unique(raw.reshape(-1, 3), axis=0).tolist() if raw.ndim == 3 else np.unique(raw).tolist()
    print('Semantic label', raw.shape, raw.dtype, '| raw values', vals[:12])
    sem = read_sem(ex['lab1'])
    print('Decoded classes:', {CLASSES[c] if c < NUM_CLASSES else 'ignore': round(float((sem == c).mean()), 3) for c in np.unique(sem)})
if ex['chg']:
    c = read_chg(ex['chg'])
    print('Change label: changed', round(float((c == 1).mean()), 3), '| unchanged', round(float((c == 0).mean()), 3),
          '| unlabeled', round(float((c == IGNORE).mean()), 3))

In [ ]:
import matplotlib.pyplot as plt


def colour(lab):
    out = np.full(lab.shape + (3,), 255, np.uint8)
    ok = lab < NUM_CLASSES
    out[ok] = PALETTE[lab[ok]]
    return out


ex_pairs = [p for p in all_pairs if p['lab1'] and p['lab2'] and p['chg']][:3]
fig, ax = plt.subplots(len(ex_pairs), 5, figsize=(20, 4 * len(ex_pairs)), squeeze=False)
for r, p in enumerate(ex_pairs):
    row = [read_image(p['img1']), read_image(p['img2']), colour(read_sem(p['lab1'])), colour(read_sem(p['lab2'])),
           read_chg(p['chg'])]
    titles = [Y1, Y2, f'labels {Y1}', f'labels {Y2}', 'change (white)']
    for a, im, t in zip(ax[r], row, titles):
        a.imshow(im, cmap='gray' if im.ndim == 2 else None, vmin=0, vmax=1)
        a.set_title(t); a.axis('off')
plt.tight_layout(); plt.show()

## 2. Fine-tune the segmentation model

In [ ]:
import random
import torch
from torch.utils.data import Dataset, DataLoader

MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)
CROP = 512


def labelled(pairs):
    """Each pair gives two training samples: one per year."""
    out = []
    for p in pairs:
        if p['lab1']: out.append((p['img1'], p['lab1']))
        if p['lab2']: out.append((p['img2'], p['lab2']))
    return out


train_pairs = [p for p in SPLITS['train'] if p['lab1'] or p['lab2']]
val_pairs = [p for p in SPLITS['val'] if p['lab1'] and p['lab2']]
if not val_pairs:
    # No labelled val split: hold out 15% of train, chosen by name so it is the same every run
    train_pairs.sort(key=lambda p: p['stem'])
    rnd = random.Random(0); rnd.shuffle(train_pairs)
    n_val = max(1, int(0.15 * len(train_pairs)))
    val_pairs, train_pairs = [p for p in train_pairs[:n_val] if p['lab1'] and p['lab2']], train_pairs[n_val:]
    print('No labelled val split, held out', len(val_pairs), 'train pairs')

test_cd = [p for p in SPLITS['test'] if p['chg']]
CD_EVAL, CD_NAME = (test_cd, 'test') if test_cd else ([p for p in val_pairs if p['chg']], 'val')
print(f'Train samples {len(labelled(train_pairs))} | val samples {len(labelled(val_pairs))} | '
      f'change detection scored on {len(CD_EVAL)} {CD_NAME} pairs')


class SegDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        ip, lp = self.samples[i]
        img, lab = read_image(ip), read_sem(lp)
        H, W = lab.shape
        if H < CROP or W < CROP:   # pad small tiles
            ph, pw = max(0, CROP - H), max(0, CROP - W)
            img = cv2.copyMakeBorder(img, 0, ph, 0, pw, cv2.BORDER_REFLECT)
            lab = cv2.copyMakeBorder(lab, 0, ph, 0, pw, cv2.BORDER_CONSTANT, value=IGNORE)
            H, W = lab.shape
        y, x = random.randint(0, H - CROP), random.randint(0, W - CROP)
        img, lab = img[y:y + CROP, x:x + CROP], lab[y:y + CROP, x:x + CROP]
        if random.random() < 0.5:
            img, lab = img[:, ::-1], lab[:, ::-1]
        if random.random() < 0.5:
            img, lab = img[::-1], lab[::-1]
        k = random.randint(0, 3)
        img, lab = np.rot90(img, k), np.rot90(lab, k)
        if random.random() < 0.5:
            a, b = random.uniform(0.8, 1.2), random.uniform(-20, 20)
            img = np.clip(img.astype(np.float32) * a + b, 0, 255)
        x = (img.astype(np.float32) / 255.0 - MEAN) / STD
        return torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1))), torch.from_numpy(np.ascontiguousarray(lab)).long()


train_dl = DataLoader(SegDataset(labelled(train_pairs)), batch_size=8, shuffle=True,
                      num_workers=2, pin_memory=True, drop_last=True)

# A T4 trains roughly 20 crops per second with this model (measured on the UAVid run)
print(f'Rough training time: {len(train_dl.dataset) / 20 / 60:.1f} min per epoch')

In [ ]:
import segmentation_models_pytorch as smp

device = 'cuda'
EPOCHS = 20

model = smp.Unet('resnet34', encoder_weights='imagenet', classes=NUM_CLASSES)
UAVID = f'{DRIVE}/models/unet_r34_best.pth'
if os.path.exists(UAVID):
    # Reuse everything except the final layer, which has a different number of classes
    old = torch.load(UAVID, map_location='cpu', weights_only=True)
    own = model.state_dict()
    keep = {k: v for k, v in old.items() if k in own and v.shape == own[k].shape}
    model.load_state_dict(keep, strict=False)
    print(f'Started from the UAVid model: {len(keep)} of {len(own)} tensors reused')
else:
    print('UAVid weights not found, starting from ImageNet encoder only')
model = model.to(device)

ce = torch.nn.CrossEntropyLoss(ignore_index=IGNORE)
dice = smp.losses.DiceLoss(mode='multiclass', ignore_index=IGNORE)
def loss_fn(out, y):
    out = out.float()
    return ce(out, y) + dice(out, y)

opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.amp.GradScaler('cuda')


def starts(total, tile, stride):
    s = list(range(0, max(total - tile, 0) + 1, stride))
    if s[-1] != max(total - tile, 0):
        s.append(total - tile)
    return s


@torch.no_grad()
def predict(img_rgb, tile=CROP, overlap=0.0):
    """Sliding-window prediction over a full tile, overlaps averaged."""
    H0, W0 = img_rgb.shape[:2]
    img = img_rgb
    if H0 < tile or W0 < tile:
        img = cv2.copyMakeBorder(img, 0, max(0, tile - H0), 0, max(0, tile - W0), cv2.BORDER_REFLECT)
    H, W = img.shape[:2]
    norm = torch.from_numpy(((img.astype(np.float32) / 255.0 - MEAN) / STD).transpose(2, 0, 1)).to(device)
    probs = torch.zeros(NUM_CLASSES, H, W, device=device)
    count = torch.zeros(1, H, W, device=device)
    stride = max(1, int(tile * (1 - overlap)))
    for y in starts(H, tile, stride):
        for x in starts(W, tile, stride):
            with torch.amp.autocast('cuda'):
                out = model(norm[:, y:y + tile, x:x + tile].unsqueeze(0))[0].float().softmax(0)
            probs[:, y:y + tile, x:x + tile] += out
            count[:, y:y + tile, x:x + tile] += 1
    return (probs / count).argmax(0)[:H0, :W0].cpu().numpy().astype(np.uint8)


def confusion(pred, gt, n):
    m = gt != IGNORE
    return np.bincount(gt[m].astype(np.int64) * n + pred[m], minlength=n * n).reshape(n, n)


def iou_from_cm(cm):
    tp = np.diag(cm).astype(float)
    return tp / np.maximum(cm.sum(0) + cm.sum(1) - tp, 1)

In [ ]:
import time

CKPT_LAST = f'{DRIVE}/models/hiucd_unet_r34_last.pth'
CKPT_BEST = f'{DRIVE}/models/hiucd_unet_r34_best.pth'
VAL_MAX = 300   # validate on a fixed subset each epoch so it does not eat training time
VAL_SAMPLES = labelled(val_pairs)
if len(VAL_SAMPLES) > VAL_MAX:
    VAL_SAMPLES = random.Random(1).sample(VAL_SAMPLES, VAL_MAX)
print('Validating on', len(VAL_SAMPLES), 'images per epoch')
start_epoch, best_miou, history = 0, 0.0, []

if os.path.exists(CKPT_LAST):
    ck = torch.load(CKPT_LAST, map_location=device, weights_only=False)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
    sched.load_state_dict(ck['sched']); scaler.load_state_dict(ck['scaler'])
    start_epoch, best_miou, history = ck['epoch'] + 1, ck['best_miou'], ck['history']
    print(f'Resuming from epoch {start_epoch + 1}')

for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()
    model.train(); train_loss = 0.0
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            out = model(x)
        loss = loss_fn(out, y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        train_loss += loss.item()
    sched.step()

    model.eval()
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), np.int64)
    for ip, lp in VAL_SAMPLES:
        cm += confusion(predict(read_image(ip)), read_sem(lp), NUM_CLASSES)
    iou = iou_from_cm(cm)
    present = cm.sum(1) > 0                 # classes that appear in val
    miou = float(iou[present].mean())
    history.append({'epoch': epoch + 1, 'train_loss': train_loss / len(train_dl), 'miou': miou,
                    'iou': iou.tolist(), 'present': present.tolist(),
                    'pix_acc': float(np.diag(cm).sum() / cm.sum())})

    if miou > best_miou:
        best_miou = miou
        torch.save(model.state_dict(), CKPT_BEST)
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(),
                'scaler': scaler.state_dict(), 'epoch': epoch, 'best_miou': best_miou, 'history': history}, CKPT_LAST)
    dt = time.time() - t0
    print(f'Epoch {epoch + 1}/{EPOCHS} | loss {history[-1]["train_loss"]:.3f} | val mIoU {miou:.3f} '
          f'| best {best_miou:.3f} | {dt / 60:.1f} min (about {dt * (EPOCHS - epoch - 1) / 60:.0f} min left)')

In [ ]:
import json

ep = [h['epoch'] for h in history]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(ep, [h['train_loss'] for h in history]); ax[0].set_title('Train loss')
ax[1].plot(ep, [h['miou'] for h in history]); ax[1].set_title('Val mIoU')
plt.savefig(f'{OUT}/training_curves.png', dpi=120, bbox_inches='tight'); plt.show()

best = max(history, key=lambda h: h['miou'])
print(f"Best epoch {best['epoch']} | mIoU {best['miou']:.3f} (classes present in val) | pixel acc {best['pix_acc']:.3f}\n")
for c, v, p in zip(CLASSES, best['iou'], best['present']):
    print(f'{c:11s} IoU {v:.3f}' if p else f'{c:11s} not in val set')
with open(f'{OUT}/seg_history.json', 'w') as f:
    json.dump(history, f, indent=1)

## 3. Change detection

Each pair is segmented twice, once per year, and a pixel counts as changed when its class differs.
Small specks of change are mostly segmentation noise, so a cleaned version removes changed regions smaller than `MIN_AREA_M2`.

Two reference numbers help read the result:
- **Label upper bound:** change computed from Hi-UCD's own semantic labels. This is the best this approach can do with a perfect segmentation model.
- **Model:** change computed from our predictions.

In [ ]:
MIN_AREA_M2 = 4.0                                   # ignore changed blobs smaller than this
MIN_PX = int(MIN_AREA_M2 / PIXEL_M ** 2)


def remove_small(mask, min_px):
    n, lab, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
    keep = np.zeros(n, bool); keep[1:] = stats[1:, cv2.CC_STAT_AREA] >= min_px
    return keep[lab]


def bin_counts(pred, gt):
    m = gt != IGNORE
    p, g = pred[m].astype(bool), gt[m] == 1
    return np.array([(p & g).sum(), (p & ~g).sum(), (~p & g).sum(), (~p & ~g).sum()], np.int64)  # tp fp fn tn


def scores(c):
    tp, fp, fn, tn = c.astype(float)
    prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return {'precision': prec, 'recall': rec, 'f1': 2 * prec * rec / max(prec + rec, 1e-9),
            'iou_changed': tp / max(tp + fp + fn, 1), 'overall_acc': (tp + tn) / max(c.sum(), 1),
            'changed_share_gt': (tp + fn) / max(c.sum(), 1)}


model.load_state_dict(torch.load(CKPT_BEST, map_location=device, weights_only=True))
model.eval()

counts = {'model': np.zeros(4, np.int64), 'model_cleaned': np.zeros(4, np.int64), 'label_upper_bound': np.zeros(4, np.int64)}
transitions = np.zeros((NUM_CLASSES, NUM_CLASSES), np.int64)     # from-class x to-class, cleaned predicted change
cached = []
t0 = time.time()
for i, p in enumerate(CD_EVAL):
    p1, p2 = predict(read_image(p['img1']), overlap=0.5), predict(read_image(p['img2']), overlap=0.5)
    gt = read_chg(p['chg'])
    raw = p1 != p2
    clean = remove_small(raw, MIN_PX)
    counts['model'] += bin_counts(raw, gt)
    counts['model_cleaned'] += bin_counts(clean, gt)
    if p['lab1'] and p['lab2']:
        l1, l2 = read_sem(p['lab1']), read_sem(p['lab2'])
        ok = (l1 != IGNORE) & (l2 != IGNORE)
        gt_ok = np.where(ok, gt, IGNORE).astype(np.uint8)
        counts['label_upper_bound'] += bin_counts(l1 != l2, gt_ok)
    transitions += np.bincount(p1[clean].astype(np.int64) * NUM_CLASSES + p2[clean],
                               minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)
    if len(cached) < 6:
        cached.append((p, p1, p2, clean, gt))
    if i == 0:
        print(f'About {(time.time() - t0) * len(CD_EVAL) / 60:.0f} min for {len(CD_EVAL)} pairs')

results = {k: scores(v) for k, v in counts.items() if v.sum() > 0}
print(f'\nChange detection on {len(CD_EVAL)} {CD_NAME} pairs ({Y1} -> {Y2}), min blob {MIN_AREA_M2} m2\n')
print(f'{"":20s} {"Precision":>9s} {"Recall":>7s} {"F1":>6s} {"IoU":>6s} {"OA":>6s}')
for k, r in results.items():
    print(f'{k:20s} {r["precision"]:9.3f} {r["recall"]:7.3f} {r["f1"]:6.3f} {r["iou_changed"]:6.3f} {r["overall_acc"]:6.3f}')
print(f'\nChanged pixels in ground truth: {100 * results["model"]["changed_share_gt"]:.1f}%')

In [ ]:
area = transitions * PIXEL_M ** 2
order = np.dstack(np.unravel_index(np.argsort(-area, axis=None), area.shape))[0]
print('Most common predicted changes (cleaned):')
top = []
for a, b in order[:10]:
    if area[a, b] <= 0:
        break
    top.append({'from': CLASSES[a], 'to': CLASSES[b], 'area_m2': float(area[a, b])})
    print(f'  {CLASSES[a]:11s} -> {CLASSES[b]:11s} {area[a, b]:10.0f} m2')

with open(f'{OUT}/change_results.json', 'w') as f:
    json.dump({'years': [Y1, Y2], 'eval_split': CD_NAME, 'pairs': len(CD_EVAL), 'min_area_m2': MIN_AREA_M2,
               'scores': results, 'top_transitions': top, 'seg_best': best}, f, indent=1)

In [ ]:
# Green = correctly found change, red = false alarm, blue = missed change
def error_map(pred, gt):
    out = np.full(gt.shape + (3,), 255, np.uint8)
    out[gt == IGNORE] = (200, 200, 200)
    out[(pred == 1) & (gt == 1)] = (0, 170, 0)
    out[(pred == 1) & (gt == 0)] = (220, 0, 0)
    out[(pred == 0) & (gt == 1)] = (0, 90, 255)
    return out


fig, ax = plt.subplots(len(cached), 5, figsize=(20, 4 * len(cached)), squeeze=False)
for r, (p, p1, p2, clean, gt) in enumerate(cached):
    row = [read_image(p['img1']), read_image(p['img2']), colour(p1), colour(p2), error_map(clean.astype(np.uint8), gt)]
    for a, im, t in zip(ax[r], row, [Y1, Y2, f'predicted {Y1}', f'predicted {Y2}', 'change: hit / false / missed']):
        a.imshow(im); a.set_title(t); a.axis('off')
plt.tight_layout()
plt.savefig(f'{OUT}/change_examples.jpg', dpi=80, bbox_inches='tight'); plt.show()
print('Saved to', OUT)